In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/bench/checkpoints/pre_cell_33.pickle

In [ ]:
%%cudf.pandas.profile
### cell 33 ###

### cell 33 optimized ###

# define the grouping of framework columns
framework_groups = {
    'TensorFlow/Keras': ['TensorFlow ', 'Keras '],
    'PyTorch/PyTorch Lightning/Fast.ai': ['PyTorch ', 'PyTorch Lightning ', 'Fast.ai '],
    'Xgboost/LightGBM/CatBoost': ['Xgboost ', 'LightGBM ', 'CatBoost ']
}

# subset, rename and drop rows with all NULLs (on GPU via cudf)
ml_frameworks_df_2022_cell45 = (
    responses_df_2022_cell10
    .filter(like=question_of_interest_cell44, axis=1)
    .rename(columns=lambda col: col.split('-')[-1].lstrip())
    .dropna(how='all')
)

# build all new grouped columns in one vectorized .assign, then drop originals
ml_frameworks_df_2022_cell45 = (
    ml_frameworks_df_2022_cell45
    .assign(
        **{
            grp: ml_frameworks_df_2022_cell45[cols]
                    .notna()
                    .any(axis=1)
                    .map({True: grp})
            for grp, cols in framework_groups.items()
        }
    )
    .drop(columns=[c for cols in framework_groups.values() for c in cols])
)[['Scikit—learn '] + list(framework_groups.keys())]

ml_frameworks_df_2022_cell45.info()